In [9]:
import sympy as sp
import sympy.physics.mechanics as me

me.init_vprinting()

# Time
t = sp.symbols('t')

# Generalized coordinates
# x, y, theta = me.dynamicsymbols('x y theta')
# xd, yd, thetad = me.dynamicsymbols('x y theta', 1)

x, y = me.dynamicsymbols('x y')
xd, yd = me.dynamicsymbols('x y', 1)

In [10]:
# Inertial frame
N = me.ReferenceFrame('N')

# Body frame
# B = me.ReferenceFrame('B')
# B.orient(N, 'Axis', [theta, N.z])
B = N

In [11]:
# Origin
O = me.Point('O')
O.set_vel(N, 0)

# Body center of mass
P = me.Point('P')
P.set_pos(O, x * N.x + y * N.y)
P.set_vel(N, xd * N.x + yd * N.y)

In [12]:
# m, Izz = sp.symbols('m Izz')
m = sp.symbols('m)')
# inertia = me.inertia(B, 0, 0, Izz)
# body = me.RigidBody('Body', P, B, m, (inertia, P))

body = me.Particle('Body', P, m)

In [13]:
k, c, g = sp.symbols('k c g')

# Spring-damper force at COM
spring_force = -k * x * N.x - k * y * N.y
damper_force = -c * xd * N.x - c * yd * N.y

gravity = -m * g * N.y

forces = [
    (P, spring_force + damper_force + gravity)
]

In [16]:
#coordinates = [x, y, theta]
#speeds = [xd, yd, thetad]

#kinematic_eqs = [
#    xd - x.diff(t),
#    yd - y.diff(t),
#    thetad - theta.diff(t)
#]

coordinates = [x, y]
speeds = [xd, yd]

kinematic_eqs = [
    xd - x.diff(t),
    yd - y.diff(t)
]

KM = me.KanesMethod(
    N,
    q_ind=coordinates,
    u_ind=speeds,
    kd_eqs=kinematic_eqs
)

fr, frstar = KM.kanes_equations([body], forces)

NonInvertibleMatrixError: Matrix det == 0; not invertible.

In [15]:
from scipy.integrate import solve_ivp
import numpy as np

params = {
    m: 1.0,
    Izz: 0.1,
    k: 50.0,
    c: 1.0,
    g: 9.81
}

# Create numerical RHS
rhs = KM.rhs()
rhs_func = sp.lambdify(
    (coordinates, speeds, list(params.keys())),
    rhs,
    'numpy'
)

def ode(t, y):
    q = y[:3]
    u = y[3:]
    return rhs_func(q, u, list(params.values())).flatten()

y0 = [0.1, 0.2, 0.1, 0, 0, 0]
t_span = (0, 10)

sol = solve_ivp(ode, t_span, y0, max_step=0.01)

NameError: name 'KM' is not defined